In [1]:
from langchain_community.llms import LlamaCpp
from langchain_community.chat_models.llamacpp import ChatLlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA

In [2]:
# Callbacks support token-wise streaming
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])
# Make sure the model path is correct for your system!
llm = ChatLlamaCpp(
    model_path="/home/ubuntu/LLM/models/llm/Mistral-7B-Instruct-v0.3-Q5_K_M.gguf",
    temperature=0.1,
    max_tokens=2000,
    n_ctx=4096,
    n_gpu_layers=35,
    top_p=1,
    callback_manager=callback_manager,
    verbose=True,  # Verbose is required to pass to the callback manager
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA A10G, compute capability 8.6, VMM: yes


llama_model_load_from_file_impl: using device CUDA0 (NVIDIA A10G) - 16678 MiB free
llama_model_loader: loaded meta data with 29 key-value pairs and 291 tensors from /home/ubuntu/LLM/models/llm/Mistral-7B-Instruct-v0.3-Q5_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = Mistral-7B-Instruct-v0.3
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 32768
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:             

In [9]:
from langchain.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

template = """[INST] <<SYS>>
You are an assistant for question-answering tasks.
        Use the following pieces of retrieved context to answer 
        the question. If you don't know the answer, say that you 
        don't know. Use three sentences maximum and keep the 
        answer concise. Answer in brazilian portuguese, always.
        \n\n
        Context: Theatro da Paz address: Avenida da Paz, Praça da República, S/N, 66017-060, Belem, Para, Brazil. contact number: +55 91 99332-3345
<</SYS>>

{user_input} [/INST]"""

prompt = PromptTemplate.from_template(template)


response = llm.invoke(prompt.format(user_input="Is there any contact number for the theatro da paz?"))
print(response)

 Sim, o número de cont

Llama.generate: 134 prefix-match hit, remaining 44 prompt tokens to eval


ato para o Te

atro da Paz é +55 91 99332-3345.

llama_perf_context_print:        load time =     185.18 ms
llama_perf_context_print: prompt eval time =     120.91 ms /    44 tokens (    2.75 ms per token,   363.91 tokens per second)
llama_perf_context_print:        eval time =     441.63 ms /    33 runs   (   13.38 ms per token,    74.72 tokens per second)
llama_perf_context_print:       total time =     596.42 ms /    77 tokens


content=' Sim, o número de contato para o Teatro da Paz é +55 91 99332-3345.' additional_kwargs={} response_metadata={'finish_reason': 'stop'} id='run-28591354-09e6-4798-b2ca-4c282d3a594b-0'


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "../models/embedding/all-mpnet-base-v2"
model_kwargs = {'device': 'cuda'}
embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs
)
# embedding_function = HuggingFaceEmbeddings(model_name="/home/ubuntu/LLM/models/embedding/all-MiniLM-L6-v2")

/home/ubuntu/LLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from langchain.document_loaders import CSVLoader
loader = CSVLoader('/home/ubuntu/LLM/data/Tourist_attractions_ROI_formatted.csv')
documents = loader.load()

In [12]:
import sys
__import__('pysqlite3')
import pysqlite3
sys.modules['sqlite3'] = sys.modules["pysqlite3"]
import chromadb
from langchain.vectorstores import Chroma
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_function
)

In [13]:
from langgraph.graph import MessagesState, StateGraph

graph_builder = StateGraph(MessagesState)

In [14]:
from langchain_core.tools import tool


@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [24]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode


# Step 1: Generate an AIMessage that may include a tool-call to be sent.
def query_or_respond(state: MessagesState):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve])
    response = llm_with_tools.invoke(state["messages"])
    # MessagesState appends messages to state instead of overwriting
    return {"messages": [response]}


# Step 2: Execute the retrieval.
tools = ToolNode([retrieve])


# Step 3: Generate a response using the retrieved content.
def generate(state: MessagesState):
    """Generate answer."""
    # Get generated ToolMessages
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    # Format into prompt
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    template = """[INST] <<SYS>>
    You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer 
            the question. If you don't know the answer, say that you 
            don't know. Use three sentences maximum and keep the 
            answer concise. Answer in brazilian portuguese, always.
            \n\n
            Context: {context}
    <</SYS>>

    {user_input} [/INST]"""

    prompt = PromptTemplate.from_template(template)
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [prompt] + conversation_messages

    # Run
    response = llm.invoke(prompt.format(context = docs_content,user_input="Is there any contact number for the theatro da paz?"))
    print(response)
    print(docs_content)
    return {"messages": [response]}

In [25]:
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: END, "tools": "tools"},
)
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


ValueError: Node `query_or_respond` already present.

In [26]:
input_message = "Hello"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Hello
 Hello! How can I help you today?

Llama.generate: 3 prefix-match hit, remaining 3 prompt tokens to eval
llama_perf_context_print:        load time =     185.18 ms
llama_perf_context_print: prompt eval time =      14.56 ms /     3 tokens (    4.85 ms per token,   206.04 tokens per second)
llama_perf_context_print:        eval time =     119.22 ms /     9 runs   (   13.25 ms per token,    75.49 tokens per second)
llama_perf_context_print:       total time =     144.27 ms /    12 tokens


================================== Ai Message ==================================

 Hello! How can I help you today?


In [27]:
input_message = "Where is the Theatro da Paz? Could you please tell me the address?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Where is the Theatro da Paz? Could you please tell me the address?
 The Theatro da Paz (Theater of Peace

Llama.generate: 3 prefix-match hit, remaining 19 prompt tokens to eval


) is located

 in Belo Horizonte, Minas Gerais, Brazil.

The exact address is:

Rua Joaquim Silveira 105 - Centro
Belo Horizonte - MG, 30100-000
Brazil

llama_perf_context_print:        load time =     185.18 ms
llama_perf_context_print: prompt eval time =      56.67 ms /    19 tokens (    2.98 ms per token,   335.30 tokens per second)
llama_perf_context_print:        eval time =    1000.49 ms /    76 runs   (   13.16 ms per token,    75.96 tokens per second)
llama_perf_context_print:       total time =    1139.80 ms /    95 tokens


================================== Ai Message ==================================

 The Theatro da Paz (Theater of Peace) is located in Belo Horizonte, Minas Gerais, Brazil.

The exact address is:

Rua Joaquim Silveira 105 - Centro
Belo Horizonte - MG, 30100-000
Brazil


In [19]:
step

{'messages': [HumanMessage(content='Where is the Theatro da Paz? Could you please tell me the address?', additional_kwargs={}, response_metadata={}, id='4393e38d-9cf1-49d3-8f71-a324fd36b052'),
  AIMessage(content=' The Theatro da Paz (Theater of Peace) is located in Belo Horizonte, Minas Gerais, Brazil.\n\nThe exact address is:\n\nRua Joaquim Silveira 143 - Sala 205\nBelo Horizonte - MG, 30130-000\nBrazil', additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-3982921f-f962-40f8-a795-b4e50501d800-0')]}

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# template = """You are a very experienced travel guide. You will suggest me places of similar type that are close to my first location. 
# You are given a context with information about the touristic places in Belem. You must use this information to answer the user's question. Use only the context given to you to give your answer. If you do not know the answer, say "I don't know".

# Context: {context}

# Question: {question}

# Answer: """

template =""" <s>[INST] You are a very experienced travel guide. You will answer questions from the user in a without any explanations!
You are given a context with information about the touristic places in Belem. You must use this information to answer the user's question. Use only the context given to you to give your answer. If you do not know the answer, say "I don't know".

For instance, a case like:
Question: Where is the Ruins of Murutucu located at?

Context:
address: Estr. da Ceasa - Curió-Utinga, Belém - PA, 66610-840, Brazilcategories/0: Tourist attractioncategories/1: categories/2: categories/3: categories/4: categories/5:  categories/6:  categoryName: Tourist attraction cid: 1.6041E+19 city: Belém countryCode: BRfid: 0x92a48ced93485ab3:0xde9d3d6e85e54079 googleFoodUrl: locatedIn: location/lat: -1.4464368location/lng: -48.4279047menu: neighborhood: Curió-Utingaphone: phoneUnformatted: placeId: ChIJs1pIk-2MpJIReUDlhW49nd4postalCode: 66610-840price: rank: 1 reserveTableUrl: reviewsCount: 169reviewsDistribution/fiveStar: 90reviewsDistribution/fourStar: 26reviewsDistribution/oneStar: 13reviewsDistribution/threeStar: 28reviewsDistribution/twoStar: 12state: State of Parástreet: Estr. Da CeasasubTitle: Ruínas do Murutucutitle: Murutucu ruinstotalScore: 4url: https://www.google.com/maps/search/?api=1&query=Murutucu%20ruins&query_place_id=ChIJs1pIk-2MpJIReUDlhW49nd4website: addressFormatted: Estr. Da Ceasa - Curió-Utinga, Belém - Pa, 66610-840, Brazilcategories/7: categories/8: Tourist attractiontitleFormatted: Murutucu RuinsstreetFormatted: Estr. Da Ceasa

Would be answered as:
[/INST]

"The address of the Ruins of Murutucu is Estr. da Ceasa - Curió-Utinga, Belém - PA, 66610-840"
</s>


[INST]
Question: {question}
Context: {context}
[/INST]
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [ ]:
rag_pipeline = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 1}),
    return_source_documents=False,
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
import time
def ask_question(question):
    start_time = time.time()
    result = rag_pipeline({"query": question})
    end_time = time.time()
    
    print(f"Answer: {result['result']}")
    print(f"Time taken: {end_time - start_time:.2f} seconds")

In [ ]:
ask_question("Which street is the Ver-o-Rio located?")